In [ ]:
import pandas as pd

In [ ]:
regenie_path = "/home/jupyter/workspaces/infectiousdiseasephewas2/results/2026-02-18_validate_regenie_pipeline_by_replicating_AoU_gwas_results/regenie_results/lupus_all_assoc_case.regenie"
ht_path = "/home/jupyter/workspaces/infectiousdiseasephewas2/data/2026-02-18_validate_regenie_pipeline_by_replicating_AoU_gwas_results/chr16_all_by_all_results_MS700_11.csv"


In [ ]:
regenie = pd.read_csv(regenie_path, delim_whitespace=True)
ht = pd.read_csv(ht_path)

In [ ]:
regenie

In [ ]:
import pandas as pd
import numpy as np

# Hail: use Pvalue_log10
hail_sub = ht[["var_id", "Pvalue_log10", "BETA", "SE"]].copy()
hail_sub.rename(columns={"Pvalue_log10": "logp_hail", "BETA": "beta_hail"}, inplace=True)

# Regenie: use LOG10P, and ID as var_id
regenie_sub = regenie[["ID", "LOG10P", "BETA", "SE", "ALLELE0", "ALLELE1"]].copy()
regenie_sub.rename(columns={"ID": "var_id", "LOG10P": "logp_regenie", "BETA": "beta_regenie"}, inplace=True)

# Merge
merged = hail_sub.merge(regenie_sub, on="var_id", how="inner")
print("Overlapping variants:", len(merged))

merged.head()

In [ ]:
merged

In [ ]:
import numpy as np
from scipy import stats  # pip install scipy if you don't have it

# --- 0. Clean up extreme / bad values (optional but recommended) ---
df_clean = merged.copy()
# --- 1. Correlation of betas and logp ---

pearson_beta = df_clean["beta_hail"].corr(df_clean["beta_regenie"], method="pearson")
spearman_beta = df_clean["beta_hail"].corr(df_clean["beta_regenie"], method="spearman")

pearson_logp = df_clean["logp_hail"].corr(df_clean["logp_regenie"], method="pearson")
spearman_logp = df_clean["logp_hail"].corr(df_clean["logp_regenie"], method="spearman")

# --- 2. Sign concordance of betas ---

sign_concordance = np.mean(
    np.sign(df_clean["beta_hail"]) == np.sign(df_clean["beta_regenie"])
)

# --- 3. (Optional) Reconstruct p and Z-scores and correlate those ---
# This part assumes logp = -log10(p). If that's true for your columns:
p_hail = 10 ** (-df_clean["logp_hail"])
p_reg  = 10 ** (-df_clean["logp_regenie"])

# Two-sided Z from p, with sign from beta
z_hail = np.sign(df_clean["beta_hail"]) * stats.norm.isf(p_hail / 2.0)
z_reg  = np.sign(df_clean["beta_regenie"]) * stats.norm.isf(p_reg / 2.0)

# Clean any inf/nan from the z's
z_df = (
    df_clean.assign(z_hail=z_hail, z_reg=z_reg)
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=["z_hail", "z_reg"])
)

pearson_z = z_df["z_hail"].corr(z_df["z_reg"], method="pearson")
spearman_z = z_df["z_hail"].corr(z_df["z_reg"], method="spearman")

# --- 4. Print summary ---

print("\n=== Agreement metrics ===")
print(f"Pearson r (betas):        {pearson_beta:.4f}")
print(f"Spearman r (betas):       {spearman_beta:.4f}")
print(f"Pearson r (logp):         {pearson_logp:.4f}")
print(f"Spearman r (logp):        {spearman_logp:.4f}")
print(f"Sign concordance (beta):  {sign_concordance:.2%}")

print(f"Pearson r (Z-scores):     {pearson_z:.4f}")
print(f"Spearman r (Z-scores):    {spearman_z:.4f}")

# --- 5. Optional: quick plots if you want visuals ---

import matplotlib.pyplot as plt

# Beta scatter
plt.figure()
plt.scatter(df_clean["beta_hail"], df_clean["beta_regenie"], s=2, alpha=0.3)
plt.xlabel("beta_hail")
plt.ylabel("beta_regenie")
plt.title("Hail vs REGENIE betas")
plt.axhline(0, linewidth=0.5)
plt.axvline(0, linewidth=0.5)
plt.show()

# Z scatter
plt.figure()
plt.scatter(z_df["z_hail"], z_df["z_reg"], s=2, alpha=0.3)
plt.xlabel("Z_hail")
plt.ylabel("Z_regenie")
plt.title("Hail vs REGENIE Z-scores")
plt.axhline(0, linewidth=0.5)
plt.axvline(0, linewidth=0.5)
plt.show()

In [ ]:
df_flipped = merged.copy()
df_flipped["beta_regenie"] = -df_flipped["beta_regenie"]

print(merged["beta_hail"].corr(df_flipped["beta_regenie"]))

In [ ]:
# Check a small slice
subset = merged.sample(20, random_state=1)

print(subset[["var_id", "beta_hail", "beta_regenie"]])
print("Sign agreement in subset:",
      np.mean(np.sign(subset["beta_hail"]) == np.sign(subset["beta_regenie"])))

In [ ]:
# Look only at strongest signals in Hail
top = merged.sort_values("logp_hail", ascending=False).head(200)

print("Top Hail beta correlation:",
      top["beta_hail"].corr(top["beta_regenie"]))

print("Top Hail sign concordance:",
      np.mean(np.sign(top["beta_hail"]) == np.sign(top["beta_regenie"])))

In [ ]:
import numpy as np

df = merged.copy()

# 1️⃣ Extract REF and ALT from var_id
split = df["var_id"].str.split(":", expand=True)
df["REF_from_varid"] = split[2]
df["ALT_from_varid"] = split[3]

# 2️⃣ Align REGENIE beta to ALT
# If ALLELE1 == ALT → keep beta
# If ALLELE1 == REF → flip beta
# Else → set to NaN (shouldn't happen if merge is correct)

df["beta_reg_aligned"] = np.where(
    df["ALLELE1"] == df["ALT_from_varid"],
    df["beta_regenie"],
    np.where(
        df["ALLELE1"] == df["REF_from_varid"],
        -df["beta_regenie"],
        np.nan
    )
)

# Drop problematic rows (if any)
df2 = df.dropna(subset=["beta_reg_aligned", "beta_hail"])


In [ ]:
df2

In [ ]:
print("Pearson r (betas):",
      df2["beta_hail"].corr(df2["beta_reg_aligned"]))

print("Spearman r (betas):",
      df2["beta_hail"].corr(df2["beta_reg_aligned"], method="spearman"))

print("Sign concordance:",
      np.mean(np.sign(df2["beta_hail"]) ==
              np.sign(df2["beta_reg_aligned"])))

In [ ]:
from scipy import stats

z_hail = df2["beta_hail"] / df2["SE_x"]
z_reg  = df2["beta_reg_aligned"] / df2["SE_y"]

print("Pearson r (Z):", np.corrcoef(z_hail, z_reg)[0,1])
print("Sign concordance (Z):",
      np.mean(np.sign(z_hail) == np.sign(z_reg)))